# Notebook 05 — Feature Engineering

This notebook converts the train, validation, and test splits into model-ready features.

Goals:
- Build features identified during EDA
- Use only information available at the prediction point
- Exclude future-information leakage
- Handle missing numerical and categorical values
- Group infrequent categorical values
- One-hot encode categorical features
- Scale numerical features
- Fit all learned transformations on the training split only
- Apply the fitted transformations to validation and test
- Save the transformed feature matrices
- Save labels, order IDs, feature names, and every fitted transformer

Prediction point:
**After order approval and before carrier handoff.**

Artifacts:
`artifacts/05_features/`

In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd

from scipy import sparse

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)


pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.width",
    220
)

print(
    "Imports loaded successfully."
)

Imports loaded successfully.


In [2]:
def find_project_root():

    current_path = Path.cwd().resolve()

    candidate_roots = [
        current_path,
        *current_path.parents,
    ]

    for candidate in candidate_roots:

        if (
            (candidate / "compose.yaml").exists()
            and
            (candidate / "requirements.txt").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the Olist-MLOps project root. "
        "Run Jupyter from inside the project directory."
    )


PROJECT_ROOT = find_project_root()


# --------------------------------------------------
# Notebook 03 split artifacts
# --------------------------------------------------

SPLITS_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "03_splits"
)

TRAIN_PATH = (
    SPLITS_DIR
    / "train.parquet"
)

VAL_PATH = (
    SPLITS_DIR
    / "validation.parquet"
)

TEST_PATH = (
    SPLITS_DIR
    / "test.parquet"
)


# --------------------------------------------------
# Notebook 04 EDA artifact
# --------------------------------------------------

EDA_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "04_eda"
)

EDA_FINDINGS_PATH = (
    EDA_DIR
    / "findings_summary.md"
)


# --------------------------------------------------
# Notebook 05 output
# --------------------------------------------------

OUTPUT_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "05_features"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# --------------------------------------------------
# Validate required artifacts
# --------------------------------------------------

assert TRAIN_PATH.exists(), (
    f"Training split not found: {TRAIN_PATH}"
)

assert VAL_PATH.exists(), (
    f"Validation split not found: {VAL_PATH}"
)

assert TEST_PATH.exists(), (
    f"Test split not found: {TEST_PATH}"
)

assert EDA_FINDINGS_PATH.exists(), (
    f"Notebook 04 EDA findings not found: {EDA_FINDINGS_PATH}"
)


print(
    "Project root :",
    PROJECT_ROOT
)

print(
    "Train        :",
    TRAIN_PATH
)

print(
    "Validation   :",
    VAL_PATH
)

print(
    "Test         :",
    TEST_PATH
)

print(
    "EDA findings :",
    EDA_FINDINGS_PATH
)

print(
    "Output       :",
    OUTPUT_DIR
)

print(
    "\nAll required artifacts found."
)

Project root : G:\(01)04\Qafza_MLOps\Olist-MLOps
Train        : G:\(01)04\Qafza_MLOps\Olist-MLOps\artifacts\03_splits\train.parquet
Validation   : G:\(01)04\Qafza_MLOps\Olist-MLOps\artifacts\03_splits\validation.parquet
Test         : G:\(01)04\Qafza_MLOps\Olist-MLOps\artifacts\03_splits\test.parquet
EDA findings : G:\(01)04\Qafza_MLOps\Olist-MLOps\artifacts\04_eda\findings_summary.md
Output       : G:\(01)04\Qafza_MLOps\Olist-MLOps\artifacts\05_features

All required artifacts found.


In [3]:
eda_findings = (
    EDA_FINDINGS_PATH
    .read_text(
        encoding="utf-8"
    )
)

print(
    "EDA findings loaded successfully."
)

print(
    "\n"
    + "=" * 70
)

print(
    "NOTEBOOK 04 — EDA FINDINGS"
)

print(
    "=" * 70
)

print()

print(
    eda_findings
)

EDA findings loaded successfully.

NOTEBOOK 04 — EDA FINDINGS

# EDA Findings Summary

## Dataset
- Training rows: 67,529
- Original training columns: 46
- Training late-delivery rate: 7.83%

## Data Quality
- Constant columns: order_status
- Highest missingness: primary_product_category (1.78%); avg_product_photos_qty (1.77%); review_count (0.76%); has_review_comment (0.76%); review_score_max (0.76%)

## Numerical Features
- Strongest numerical correlations with the target: avg_freight_value (0.072); total_freight_value (0.050); avg_product_weight_g (0.028); avg_item_price (0.028); max_product_weight_g (0.027)
- Most skewed numerical variables: payment_records (22.42); unique_seller_states (15.18); total_item_price (10.72); unique_sellers (10.52); payment_total (10.15)
- IQR-based outliers were measured but were not automatically removed.

## Temporal Patterns
- Highest observed monthly late rate: 2018-03 (18.96%).
- Late-delivery rates vary across time periods, supporting the use of 

In [4]:
train = pd.read_parquet(
    TRAIN_PATH
)

validation = pd.read_parquet(
    VAL_PATH
)

test = pd.read_parquet(
    TEST_PATH
)

print(
    "Train      :",
    train.shape
)

print(
    "Validation :",
    validation.shape
)

print(
    "Test       :",
    test.shape
)

Train      : (67529, 46)
Validation : (14470, 46)
Test       : (14471, 46)


In [5]:
splits = {
    "Train": train,
    "Validation": validation,
    "Test": test,
}

for name, data in splits.items():

    duplicate_orders = (
        data["order_id"]
        .duplicated()
        .sum()
    )

    missing_labels = (
        data["is_late"]
        .isna()
        .sum()
    )

    print(
        f"{name:<12} "
        f"rows={len(data):,} | "
        f"columns={data.shape[1]} | "
        f"duplicates={duplicate_orders} | "
        f"missing_labels={missing_labels}"
    )

    assert duplicate_orders == 0
    assert missing_labels == 0


train_columns = set(
    train.columns
)

assert (
    set(validation.columns)
    == train_columns
)

assert (
    set(test.columns)
    == train_columns
)

print(
    "\nInput split validation passed."
)

Train        rows=67,529 | columns=46 | duplicates=0 | missing_labels=0
Validation   rows=14,470 | columns=46 | duplicates=0 | missing_labels=0
Test         rows=14,471 | columns=46 | duplicates=0 | missing_labels=0

Input split validation passed.


## Feature Engineering Rules

The model prediction point is assumed to be after order approval and before carrier handoff.

Therefore:

### Allowed
Information that is known by that time, such as:
- Purchase information
- Approval information
- Order-item aggregates
- Product characteristics
- Seller/customer geography
- Payment information
- Estimated delivery date

### Excluded as leakage
Information generated after the prediction point:
- Actual customer delivery date
- Carrier handoff date
- Review scores and comments
- `delay_days`
- Any feature derived from actual delivery outcome

### Training rule
Any transformation that learns from the data must be fitted on the training split only.

Validation and test are transformed using the already-fitted training objects.

In [6]:
FIXED_HOLIDAY_MONTH_DAYS = {
    (1, 1),
    (4, 21),
    (5, 1),
    (9, 7),
    (10, 12),
    (11, 2),
    (11, 15),
    (12, 25),
}


def engineer_features(df):

    data = df.copy()

    date_columns = [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_estimated_delivery_date",
    ]

    for column in date_columns:

        data[column] = pd.to_datetime(
            data[column],
            errors="coerce"
        )

    purchase_time = (
        data[
            "order_purchase_timestamp"
        ]
    )

    # -------------------------
    # Time features
    # -------------------------

    data[
        "purchase_month"
    ] = (
        purchase_time
        .dt.month
        .astype("Int64")
        .astype("string")
    )

    data[
        "purchase_weekday"
    ] = (
        purchase_time
        .dt.day_name()
        .astype("string")
    )

    data[
        "purchase_hour"
    ] = (
        purchase_time
        .dt.hour
    )

    data[
        "estimated_delivery_days"
    ] = (
        data[
            "order_estimated_delivery_date"
        ]
        - purchase_time
    ).dt.total_seconds() / 86400

    data[
        "approval_hours"
    ] = (
        data[
            "order_approved_at"
        ]
        - purchase_time
    ).dt.total_seconds() / 3600

    # -------------------------
    # Fixed national holiday
    # -------------------------

    month_day = list(
        zip(
            purchase_time.dt.month,
            purchase_time.dt.day,
        )
    )

    data[
        "is_fixed_national_holiday"
    ] = [
        int(value in FIXED_HOLIDAY_MONTH_DAYS)
        if not (
            pd.isna(value[0])
            or pd.isna(value[1])
        )
        else np.nan
        for value in month_day
    ]

    # -------------------------
    # Same customer/seller state
    # -------------------------

    valid_states = (
        data[
            "customer_state"
        ].notna()
        &
        data[
            "primary_seller_state"
        ].notna()
    )

    data[
        "same_customer_seller_state"
    ] = np.nan

    data.loc[
        valid_states,
        "same_customer_seller_state"
    ] = (
        data.loc[
            valid_states,
            "customer_state"
        ]
        ==
        data.loc[
            valid_states,
            "primary_seller_state"
        ]
    ).astype(int)

    # -------------------------
    # Customer-seller distance
    # -------------------------

    customer_lat = np.radians(
        data[
            "customer_lat"
        ]
    )

    customer_lng = np.radians(
        data[
            "customer_lng"
        ]
    )

    seller_lat = np.radians(
        data[
            "avg_seller_lat"
        ]
    )

    seller_lng = np.radians(
        data[
            "avg_seller_lng"
        ]
    )

    delta_lat = (
        seller_lat
        - customer_lat
    )

    delta_lng = (
        seller_lng
        - customer_lng
    )

    a = (
        np.sin(
            delta_lat / 2
        ) ** 2
        +
        np.cos(
            customer_lat
        )
        *
        np.cos(
            seller_lat
        )
        *
        np.sin(
            delta_lng / 2
        ) ** 2
    )

    c = (
        2
        * np.arcsin(
            np.sqrt(a)
        )
    )

    data[
        "customer_seller_distance_km"
    ] = (
        6371.0 * c
    )

    return data

In [7]:
train_engineered = (
    engineer_features(
        train
    )
)

validation_engineered = (
    engineer_features(
        validation
    )
)

test_engineered = (
    engineer_features(
        test
    )
)

assert (
    train_engineered["order_id"]
    .equals(train["order_id"])
)

assert (
    validation_engineered["order_id"]
    .equals(validation["order_id"])
)

assert (
    test_engineered["order_id"]
    .equals(test["order_id"])
)

assert (
    train_engineered["is_late"]
    .equals(train["is_late"])
)

assert (
    validation_engineered["is_late"]
    .equals(validation["is_late"])
)

assert (
    test_engineered["is_late"]
    .equals(test["is_late"])
)

print(
    "Row order and labels preserved."
)

print(
    "Train engineered      :",
    train_engineered.shape
)

print(
    "Validation engineered :",
    validation_engineered.shape
)

print(
    "Test engineered       :",
    test_engineered.shape
)

Row order and labels preserved.
Train engineered      : (67529, 54)
Validation engineered : (14470, 54)
Test engineered       : (14471, 54)


In [8]:
NUMERIC_FEATURES = [
    "item_count",
    "unique_products",
    "unique_sellers",

    "total_item_price",
    "avg_item_price",

    "total_freight_value",
    "avg_freight_value",

    "unique_product_categories",

    "avg_product_weight_g",
    "max_product_weight_g",

    "avg_product_length_cm",
    "avg_product_height_cm",
    "avg_product_width_cm",

    "avg_product_photos_qty",

    "unique_seller_states",

    "payment_records",
    "payment_types_count",
    "payment_total",
    "payment_installments_max",

    # Engineered
    "purchase_hour",
    "estimated_delivery_days",
    "approval_hours",
    "customer_seller_distance_km",
    "same_customer_seller_state",
    "is_fixed_national_holiday",
]


CATEGORICAL_FEATURES = [
    "customer_state",
    "primary_product_category",
    "primary_seller_state",
    "primary_payment_type",

    # Engineered
    "purchase_month",
    "purchase_weekday",
]


FINAL_FEATURES = (
    NUMERIC_FEATURES
    + CATEGORICAL_FEATURES
)


print(
    "Numeric features:",
    len(NUMERIC_FEATURES)
)

print(
    "Categorical features:",
    len(CATEGORICAL_FEATURES)
)

print(
    "Raw final features:",
    len(FINAL_FEATURES)
)

Numeric features: 25
Categorical features: 6
Raw final features: 31


In [9]:
required_columns = (
    FINAL_FEATURES
    + [
        "order_id",
        "is_late",
    ]
)


for name, data in {
    "Train":
        train_engineered,

    "Validation":
        validation_engineered,

    "Test":
        test_engineered,
}.items():

    missing_columns = [
        column
        for column in required_columns
        if column not in data.columns
    ]

    assert not missing_columns, (
        f"{name} missing columns: "
        f"{missing_columns}"
    )


LEAKAGE_COLUMNS = {
    "order_delivered_carrier_date",
    "order_delivered_customer_date",

    "review_count",
    "review_score_mean",
    "review_score_min",
    "review_score_max",
    "has_review_comment",

    "delay_days",
    "actual_delivery_days",
}


leaked_features = (
    set(FINAL_FEATURES)
    & LEAKAGE_COLUMNS
)

print(
    "Leakage features selected:",
    leaked_features
)

assert not leaked_features

print(
    "\nFeature availability and leakage validation passed."
)

Leakage features selected: set()

Feature availability and leakage validation passed.


In [10]:
X_train_raw = (
    train_engineered[
        FINAL_FEATURES
    ]
    .copy()
)

X_validation_raw = (
    validation_engineered[
        FINAL_FEATURES
    ]
    .copy()
)

X_test_raw = (
    test_engineered[
        FINAL_FEATURES
    ]
    .copy()
)


y_train = (
    train_engineered[
        "is_late"
    ]
    .astype("int8")
    .to_numpy()
)

y_validation = (
    validation_engineered[
        "is_late"
    ]
    .astype("int8")
    .to_numpy()
)

y_test = (
    test_engineered[
        "is_late"
    ]
    .astype("int8")
    .to_numpy()
)


train_order_ids = (
    train_engineered[
        "order_id"
    ]
    .copy()
)

validation_order_ids = (
    validation_engineered[
        "order_id"
    ]
    .copy()
)

test_order_ids = (
    test_engineered[
        "order_id"
    ]
    .copy()
)


print(
    "X_train raw      :",
    X_train_raw.shape
)

print(
    "X_validation raw :",
    X_validation_raw.shape
)

print(
    "X_test raw       :",
    X_test_raw.shape
)

X_train raw      : (67529, 31)
X_validation raw : (14470, 31)
X_test raw       : (14471, 31)


In [11]:
selected_missing = (
    X_train_raw
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
)

selected_missing = (
    selected_missing[
        selected_missing > 0
    ]
)

display(
    selected_missing
)

primary_product_category       1201
avg_product_photos_qty         1197
customer_seller_distance_km     344
max_product_weight_g             16
avg_product_width_cm             16
avg_product_height_cm            16
avg_product_length_cm            16
avg_product_weight_g             16
approval_hours                   14
payment_types_count               1
payment_installments_max          1
payment_records                   1
primary_payment_type              1
payment_total                     1
dtype: int64

In [12]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
            ),
        ),

        (
            "scaler",
            StandardScaler(),
        ),
    ]
)


categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="__MISSING__",
            ),
        ),

        (
            "encoder",
          OneHotEncoder(
                handle_unknown="infrequent_if_exist",
                min_frequency=50,
                sparse_output=True,
            ),
        ),
    ]
)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            NUMERIC_FEATURES,
        ),

        (
            "categorical",
            categorical_pipeline,
            CATEGORICAL_FEATURES,
        ),
    ],

    remainder="drop",

    sparse_threshold=1.0,
)

In [13]:
X_train = (
    preprocessor
    .fit_transform(
        X_train_raw
    )
)

print(
    "Preprocessor fitted on TRAIN ONLY."
)

print(
    "Transformed train shape:",
    X_train.shape
)

Preprocessor fitted on TRAIN ONLY.
Transformed train shape: (67529, 159)


In [14]:
X_validation = (
    preprocessor
    .transform(
        X_validation_raw
    )
)

X_test = (
    preprocessor
    .transform(
        X_test_raw
    )
)

print(
    "Train      :",
    X_train.shape
)

print(
    "Validation :",
    X_validation.shape
)

print(
    "Test       :",
    X_test.shape
)

Train      : (67529, 159)
Validation : (14470, 159)
Test       : (14471, 159)


In [15]:
feature_names = (
    preprocessor
    .get_feature_names_out()
    .tolist()
)

print(
    "Final transformed features:",
    len(feature_names)
)

print(
    "\nFirst 30 features:"
)

for feature in feature_names[:30]:

    print(
        "-",
        feature
    )

Final transformed features: 159

First 30 features:
- numeric__item_count
- numeric__unique_products
- numeric__unique_sellers
- numeric__total_item_price
- numeric__avg_item_price
- numeric__total_freight_value
- numeric__avg_freight_value
- numeric__unique_product_categories
- numeric__avg_product_weight_g
- numeric__max_product_weight_g
- numeric__avg_product_length_cm
- numeric__avg_product_height_cm
- numeric__avg_product_width_cm
- numeric__avg_product_photos_qty
- numeric__unique_seller_states
- numeric__payment_records
- numeric__payment_types_count
- numeric__payment_total
- numeric__payment_installments_max
- numeric__purchase_hour
- numeric__estimated_delivery_days
- numeric__approval_hours
- numeric__customer_seller_distance_km
- numeric__same_customer_seller_state
- numeric__is_fixed_national_holiday
- numeric__missingindicator_avg_product_weight_g
- numeric__missingindicator_max_product_weight_g
- numeric__missingindicator_avg_product_length_cm
- numeric__missingindicator

In [16]:
assert (
    X_train.shape[0]
    == len(y_train)
)

assert (
    X_validation.shape[0]
    == len(y_validation)
)

assert (
    X_test.shape[0]
    == len(y_test)
)


assert (
    X_train.shape[1]
    == X_validation.shape[1]
    == X_test.shape[1]
)


assert (
    X_train.shape[1]
    == len(feature_names)
)


print(
    "Train sparse:",
    sparse.issparse(
        X_train
    )
)

print(
    "Validation sparse:",
    sparse.issparse(
        X_validation
    )
)

print(
    "Test sparse:",
    sparse.issparse(
        X_test
    )
)

print(
    "\nFinal matrix validation passed."
)

Train sparse: True
Validation sparse: True
Test sparse: True

Final matrix validation passed.


In [17]:
for name, matrix in {
    "Train":
        X_train,

    "Validation":
        X_validation,

    "Test":
        X_test,
}.items():

    values = (
        matrix.data
        if sparse.issparse(matrix)
        else matrix.ravel()
    )

    nan_count = int(
        np.isnan(values)
        .sum()
    )

    inf_count = int(
        np.isinf(values)
        .sum()
    )

    print(
        f"{name:<12} "
        f"NaN={nan_count} | "
        f"Inf={inf_count}"
    )

    assert nan_count == 0
    assert inf_count == 0


print(
    "\nNo NaN or infinite values in transformed matrices."
)

Train        NaN=0 | Inf=0
Validation   NaN=0 | Inf=0
Test         NaN=0 | Inf=0

No NaN or infinite values in transformed matrices.


In [18]:
train_engineered_output = (
    X_train_raw.copy()
)

train_engineered_output.insert(
    0,
    "order_id",
    train_order_ids.values
)

train_engineered_output[
    "is_late"
] = y_train


validation_engineered_output = (
    X_validation_raw.copy()
)

validation_engineered_output.insert(
    0,
    "order_id",
    validation_order_ids.values
)

validation_engineered_output[
    "is_late"
] = y_validation


test_engineered_output = (
    X_test_raw.copy()
)

test_engineered_output.insert(
    0,
    "order_id",
    test_order_ids.values
)

test_engineered_output[
    "is_late"
] = y_test


train_engineered_output.to_parquet(
    OUTPUT_DIR
    / "engineered_train.parquet",
    index=False
)

validation_engineered_output.to_parquet(
    OUTPUT_DIR
    / "engineered_validation.parquet",
    index=False
)

test_engineered_output.to_parquet(
    OUTPUT_DIR
    / "engineered_test.parquet",
    index=False
)

print(
    "Engineered raw feature tables saved."
)

Engineered raw feature tables saved.


In [19]:
sparse.save_npz(
    OUTPUT_DIR
    / "X_train.npz",
    X_train.tocsr()
)

sparse.save_npz(
    OUTPUT_DIR
    / "X_validation.npz",
    X_validation.tocsr()
)

sparse.save_npz(
    OUTPUT_DIR
    / "X_test.npz",
    X_test.tocsr()
)

print(
    "Transformed feature matrices saved."
)

Transformed feature matrices saved.


In [20]:
np.save(
    OUTPUT_DIR
    / "y_train.npy",
    y_train
)

np.save(
    OUTPUT_DIR
    / "y_validation.npy",
    y_validation
)

np.save(
    OUTPUT_DIR
    / "y_test.npy",
    y_test
)

print(
    "Labels saved."
)

Labels saved.


In [21]:
pd.DataFrame({
    "order_id":
        train_order_ids
}).to_csv(
    OUTPUT_DIR
    / "train_order_ids.csv",
    index=False
)


pd.DataFrame({
    "order_id":
        validation_order_ids
}).to_csv(
    OUTPUT_DIR
    / "validation_order_ids.csv",
    index=False
)


pd.DataFrame({
    "order_id":
        test_order_ids
}).to_csv(
    OUTPUT_DIR
    / "test_order_ids.csv",
    index=False
)


print(
    "Order IDs saved."
)

Order IDs saved.


In [22]:
PREPROCESSOR_PATH = (
    OUTPUT_DIR
    / "preprocessor.joblib"
)

joblib.dump(
    preprocessor,
    PREPROCESSOR_PATH
)

print(
    "Fitted preprocessor saved:"
)

print(
    PREPROCESSOR_PATH
)

Fitted preprocessor saved:
G:\(01)04\Qafza_MLOps\Olist-MLOps\artifacts\05_features\preprocessor.joblib


In [23]:
FEATURE_NAMES_PATH = (
    OUTPUT_DIR
    / "feature_names.json"
)

FEATURE_NAMES_PATH.write_text(
    json.dumps(
        feature_names,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8"
)

print(
    "Feature names saved:"
)

print(
    FEATURE_NAMES_PATH
)

Feature names saved:
G:\(01)04\Qafza_MLOps\Olist-MLOps\artifacts\05_features\feature_names.json


In [24]:
feature_config = {
    "prediction_point":
        (
            "after order approval and "
            "before carrier handoff"
        ),

    "target":
        "is_late",

    "id_column":
        "order_id",

    "numeric_features":
        NUMERIC_FEATURES,

    "categorical_features":
        CATEGORICAL_FEATURES,

    "raw_feature_count":
        len(FINAL_FEATURES),

    "transformed_feature_count":
        len(feature_names),

    "excluded_high_cardinality_features": [
        "customer_city",
        "customer_zip_code_prefix",
    ],

    "excluded_leakage_features":
        sorted(
            LEAKAGE_COLUMNS
        ),

    "numeric_missing_strategy":
        "median + missing indicators",

    "categorical_missing_strategy":
        "__MISSING__ category",

    "categorical_encoding":
        "one-hot encoding",

    "rare_category_rule":
        "min_frequency=50",

    "numeric_scaling":
        "StandardScaler",

    "fit_rule":
        (
            "All learned transformations "
            "fit on training split only"
        ),
}


FEATURE_CONFIG_PATH = (
    OUTPUT_DIR
    / "feature_config.json"
)

FEATURE_CONFIG_PATH.write_text(
    json.dumps(
        feature_config,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8"
)

print(
    "Feature configuration saved."
)

Feature configuration saved.


In [25]:
loaded_preprocessor = (
    joblib.load(
        PREPROCESSOR_PATH
    )
)

loaded_X_train = (
    sparse.load_npz(
        OUTPUT_DIR
        / "X_train.npz"
    )
)

loaded_X_validation = (
    sparse.load_npz(
        OUTPUT_DIR
        / "X_validation.npz"
    )
)

loaded_X_test = (
    sparse.load_npz(
        OUTPUT_DIR
        / "X_test.npz"
    )
)

loaded_y_train = np.load(
    OUTPUT_DIR
    / "y_train.npy"
)

loaded_y_validation = np.load(
    OUTPUT_DIR
    / "y_validation.npy"
)

loaded_y_test = np.load(
    OUTPUT_DIR
    / "y_test.npy"
)


print(
    "Loaded Train      :",
    loaded_X_train.shape
)

print(
    "Loaded Validation :",
    loaded_X_validation.shape
)

print(
    "Loaded Test       :",
    loaded_X_test.shape
)

assert (
    loaded_X_train.shape
    == X_train.shape
)

assert (
    loaded_X_validation.shape
    == X_validation.shape
)

assert (
    loaded_X_test.shape
    == X_test.shape
)

assert (
    len(loaded_y_train)
    == loaded_X_train.shape[0]
)

assert (
    len(loaded_y_validation)
    == loaded_X_validation.shape[0]
)

assert (
    len(loaded_y_test)
    == loaded_X_test.shape[0]
)

print(
    "\nSaved feature artifacts validated successfully."
)

Loaded Train      : (67529, 159)
Loaded Validation : (14470, 159)
Loaded Test       : (14471, 159)

Saved feature artifacts validated successfully.


In [26]:
print(
    "Feature engineering artifacts:"
)

for path in sorted(
    OUTPUT_DIR.rglob("*")
):

    if path.is_file():

        print(
            "-",
            path.relative_to(
                OUTPUT_DIR
            )
        )

Feature engineering artifacts:
- engineered_test.parquet
- engineered_train.parquet
- engineered_validation.parquet
- feature_config.json
- feature_names.json
- preprocessor.joblib
- test_order_ids.csv
- train_order_ids.csv
- validation_order_ids.csv
- X_test.npz
- X_train.npz
- X_validation.npz
- y_test.npy
- y_train.npy
- y_validation.npy


In [27]:
print(
    "=" * 70
)

print(
    "NOTEBOOK 05 COMPLETE"
)

print(
    "=" * 70)

print(
    f"Raw selected features       : "
    f"{len(FINAL_FEATURES)}"
)

print(
    f"Final transformed features  : "
    f"{len(feature_names)}"
)

print()

print(
    f"Train matrix                : "
    f"{X_train.shape}"
)

print(
    f"Validation matrix           : "
    f"{X_validation.shape}"
)

print(
    f"Test matrix                 : "
    f"{X_test.shape}"
)

print()

print(
    "Preprocessor fit on        : TRAIN ONLY"
)

print(
    "Validation transformation : TRANSFORM ONLY"
)

print(
    "Test transformation       : TRANSFORM ONLY"
)

print()

print(
    f"Preprocessor artifact       : "
    f"{PREPROCESSOR_PATH}"
)

print(
    f"Feature list artifact       : "
    f"{FEATURE_NAMES_PATH}"
)

print(
    f"Feature config artifact     : "
    f"{FEATURE_CONFIG_PATH}"
)

NOTEBOOK 05 COMPLETE
Raw selected features       : 31
Final transformed features  : 159

Train matrix                : (67529, 159)
Validation matrix           : (14470, 159)
Test matrix                 : (14471, 159)

Preprocessor fit on        : TRAIN ONLY
Validation transformation : TRANSFORM ONLY
Test transformation       : TRANSFORM ONLY

Preprocessor artifact       : G:\(01)04\Qafza_MLOps\Olist-MLOps\artifacts\05_features\preprocessor.joblib
Feature list artifact       : G:\(01)04\Qafza_MLOps\Olist-MLOps\artifacts\05_features\feature_names.json
Feature config artifact     : G:\(01)04\Qafza_MLOps\Olist-MLOps\artifacts\05_features\feature_config.json
